# 16. Signal Processing & Mathematical Transforms (5+ Years Interview Guide)
Exhaustive revision guide to Fast Fourier Transforms (np.fft.fft, ifft), frequency decomposition, and polynomial regression (np.polyfit) on transaction time series.

### Key 5-Year Interview Concepts Covered:
- **Fast Fourier Transforms**: Dedicated cell for `np.fft.fft()`, `np.fft.ifft()`, and `np.fft.fftfreq()`.
- **Curve Fitting & Polynomials**: Dedicated cell for `np.polyfit()` and `np.polyval()`.

This interactive revision guide loads and operates directly on `data/raw_transactions.csv` using dedicated cells per method.

In [ ]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

### 1D Fast Fourier Transform with `np.fft.fft()`
**Explanation**: Decomposes daily transaction volume into cyclical frequency components.

**Syntax**: `np.fft.fft(daily_signal)`

In [ ]:
daily_signal = amounts[:512]  # Power of 2 for optimal FFT
fft_coeffs = np.fft.fft(daily_signal)
print('FFT Complex Coefficients (first 3):', fft_coeffs[:3])

### Sample Frequencies with `np.fft.fftfreq()`
**Explanation**: Identifies dominant cyclical spending periodicities.

**Syntax**: `np.fft.fftfreq(len(daily_signal))`

In [ ]:
freqs = np.fft.fftfreq(len(daily_signal))
peak_idx = np.argmax(np.abs(fft_coeffs[1:len(daily_signal)//2])) + 1
print(f'Dominant Frequency Detected: {freqs[peak_idx]:.4f} cycles/transaction')

### Inverse FFT Denoising with `np.fft.ifft()`
**Explanation**: Denoises the transaction volume curve by zeroing high-frequency noise and reconstructing the signal.

**Syntax**: `np.fft.ifft(filtered_fft).real`

In [ ]:
filtered_fft = fft_coeffs.copy()
filtered_fft[np.abs(freqs) > 0.1] = 0.0  # Zero high frequencies
denoised_ts = np.fft.ifft(filtered_fft).real
print('Denoised Transaction Signal Head:', denoised_ts[:4].round(2))

### Least-Squares Polynomial Fitting with `np.polyfit()`
**Explanation**: Fits a linear regression trend line to transaction amount growth over sequential index steps.

**Syntax**: `np.polyfit(steps, amounts[:100], deg=1)`

In [ ]:
steps = np.arange(100)
linear_fit = np.polyfit(steps, amounts[:100], deg=1)
print('Fitted Linear Trend Slope & Intercept [m, c]:', linear_fit.round(4))

### Polynomial Evaluation with `np.polyval()`
**Explanation**: Forecasts future transaction amounts using the fitted polynomial equation.

**Syntax**: `np.polyval(linear_fit, future_steps)`

In [ ]:
future_steps = np.array([101, 102, 103])
predictions = np.polyval(linear_fit, future_steps)
print('Forecasted Transaction Amounts for Next 3 Steps:', predictions.round(2))

## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Quadratic Polynomial Trendline on Account Spending vs Age
**Explanation**: Fit a 2nd-degree polynomial curve ($y = ax^2 + bx + c$) modeling transaction amount as a non-linear function of account age.

**Syntax**: `np.polyfit(account_ages[:500], amounts[:500], deg=2)`

In [ ]:
quad_coeffs = np.polyfit(account_ages[:500], amounts[:500], deg=2)
residuals = amounts[:500] - np.polyval(quad_coeffs, account_ages[:500])
print('Quadratic Polynomial Coefficients [a, b, c]:', quad_coeffs.round(4))
print('Root Mean Squared Error (RMSE):', round(np.sqrt(np.mean(residuals**2)), 2))